<div style="background:#082956; color:#ffffff; border-radius:18px; padding:28px 32px; font-family:Sora, Manrope, system-ui, -apple-system, 'Segoe UI', sans-serif">
<div style="font-family:'Space Mono', 'IBM Plex Mono', ui-monospace, monospace; font-size:13px; letter-spacing:.08em; color:#f4b718">III SEPEC · ENSINO · PESQUISA · EXTENSÃO · CULTURA</div>
<div style="font-size:34px; font-weight:800; line-height:1.2; margin:10px 0 8px">Animando o Cálculo com o <span style="background:#f06418; padding:0 10px; border-radius:8px">Manim</span></div>
<div style="font-size:18px; opacity:.9">Dia 2 · sexta, 27/11 · 14:00 às 17:00 · Laboratório K04-s<br>gráficos, valores dinâmicos e oficina</div>
<div style="margin-top:20px; font-size:15px; opacity:.85">Wilia Alberto e Vicente Sobrinho · UFCA, Campus Juazeiro do Norte</div><div style="display:flex; height:6px; border-radius:3px; overflow:hidden; margin-top:18px"><div style="flex:1; background:#57a52d"></div><div style="flex:1; background:#146dcc"></div><div style="flex:1; background:#f06418"></div><div style="flex:1; background:#f4b718"></div></div>
</div>

## <span style="color:#082956">■</span> Roteiro do Dia 2

* **Parte 4:** gráficos (`Axes`, `plot`), valores dinâmicos (`ValueTracker`, `always_redraw`, updaters) e exemplos de derivada e integral (14:00 – 15:15)
* **Parte 5:** estudo de caso e **oficina**: animando questões de Cálculo (15:15 – 17:00)


## <span style="color:#082956">■</span> Antes de começar

Sessão nova do Colab? Execute a instalação (uma vez por sessão) e depois a importação:

In [6]:
%%capture
# %%capture -> esconde a saída longa da instalação (apague esta linha se quiser acompanhar o progresso).
# POR QUÊ instalar? Cada sessão do Colab é uma máquina Linux "zerada": o Manim e o LaTeX não vêm instalados.
# Execute esta célula UMA vez por sessão (leva alguns minutos). Se o Colab pedir, reinicie a sessão e continue.

# Bibliotecas do sistema: cairo e pango (desenho e texto), ffmpeg (gera o vídeo) e LaTeX (fórmulas do MathTex).
!sudo apt-get update -qq
!sudo apt-get install -y -qq libcairo2-dev libpango1.0-dev ffmpeg texlive texlive-latex-extra texlive-fonts-extra texlive-latex-recommended texlive-science tipa

# O próprio Manim, instalado com o pip (o "!" executa o comando no terminal da máquina, não em Python).
!pip install -q manim

In [7]:
from manim import *          # traz para o caderno as classes do Manim (Scene, MathTex, Write, Axes, ...); sem isso, nada funciona

config.media_width = "75%"   # largura do vídeo exibido no caderno (sem isso o vídeo ocupa a tela toda)
config.media_embed = True    # embute o vídeo na saída da célula; é isso que faz o vídeo aparecer no Colab
config.progress_bar = "none" # esconde as barras de progresso, que só poluem a saída

# Cores da III SEPEC (sepec.com.br), as mesmas do Dia 1: use como qualquer cor do Manim, por exemplo color=SEPEC_LARANJA
SEPEC_VERDE   = "#57a52d"
SEPEC_AZUL    = "#4a9ae8"     # versão mais clara do azul do evento (#146dcc), para contrastar com o fundo escuro
SEPEC_LARANJA = "#f06418"
SEPEC_AMARELO = "#f4b718"
SEPEC_TINTA   = "#082956"     # azul-marinho do evento
config.background_color = SEPEC_TINTA   # fundo de TODAS as cenas (o padrão do Manim é preto)


# Parte 4: gráficos e valores dinâmicos

<div style="border-left:10px solid #146dcc; background:#eaf3ff; color:#082956; border-radius:12px; padding:14px 20px; font-family:Sora, Manrope, system-ui, -apple-system, 'Segoe UI', sans-serif; font-size:17px">Eixos, funções, valores que variam, secante → tangente, Riemann e o gráfico da derivada.</div>

## <span style="color:#146dcc">■</span> Eixos e gráficos

* `Axes` cria um sistema de eixos, com intervalos `[mínimo, máximo, passo]`
* `eixos.plot(f)` desenha o gráfico de uma função `f` do Python
* `eixos.c2p(x, y)` (*coordinates to point*) converte coordenadas **dos eixos** em um ponto **da tela**

Função de trabalho desta parte:
$$
f(x) = \frac{x^3}{3} - x, \qquad f'(x) = x^2 - 1
$$

In [8]:
%%manim -qm -v WARNING Eixos
class Eixos(Scene):
    def construct(self):
        eixos = Axes(
            x_range=[-5, 5, 1],                        # [mínimo, máximo, passo das marcas] no eixo x
            y_range=[-5, 5, 1],                        # idem no eixo y
            x_length=5.5,                                # comprimento do eixo x NA TELA (unidades da tela)
            y_length=5.5,                                # comprimento do eixo y na tela
            axis_config={"include_numbers": True, "font_size": 16,    # escreve os números nas marcas, em fonte pequena
                         "tip_width": 0.15, "tip_height": 0.15}       # pontas de seta menores (o padrão é cerca de 0,35)
        )
        rotulos = eixos.get_axis_labels(x_label="x", y_label="y")    # nomes dos eixos

        def f(x):                                      # a função matemática, como função Python
            return x**3/3 - x

        # plot usa a função f; x_range limita o domínio desenhado (evita sair dos eixos)
        grafico = eixos.plot(f, x_range=[-2.5, 2.5], color=YELLOW)
        rotulo_f = MathTex(r"f(x) = \frac{x^3}{3} - x", color=YELLOW).scale(0.8).to_corner(UL)
        self.play(Create(eixos), Write(rotulos))
        self.play(Create(grafico), Write(rotulo_f), run_time=2)

        # c2p converte (x, y) dos EIXOS em ponto da TELA: sem isso o ponto iria para o lugar errado
        ponto = Dot(eixos.c2p(1, f(1)), color=RED)     # ponto (1, f(1)) = (1, -2/3)
        self.play(FadeIn(ponto))
        self.wait()

Manim Community v0.22.0

## <span style="color:#146dcc">■</span> `Axes` em detalhe

| parâmetro / método | efeito |
|---|---|
| `x_range=[min, max, passo]`, `y_range=[...]` | intervalo e distância entre as marcas |
| `x_length`, `y_length` | tamanho **na tela** (independe do intervalo) |
| `tips=False` | tira as pontas de seta dos eixos |
| `axis_config={...}` | estilo dos dois eixos: `include_numbers`, `font_size`, `color`, `numbers_to_include`, ... |
| `x_axis_config={...}`, `y_axis_config={...}` | estilo de **um** eixo só |
| `eixos.add_coordinates()` | números nos dois eixos (alternativa ao `include_numbers`) |
| `eixos.plot(f, x_range=[a, b], color=...)` | gráfico de `f` (sempre limite `x_range`!) |
| `eixos.c2p(x, y)` / `eixos.p2c(ponto)` | coordenadas dos eixos ⇄ ponto da tela |
| `eixos.i2gp(x, grafico)` | ponto do **gráfico** com abscissa `x` |
| `eixos.get_graph_label(grafico, "f(x)", x_val=2)` | rótulo colado na curva |
| `eixos.get_vertical_line(ponto)`, `get_horizontal_line(ponto)` | linhas do eixo até o ponto |
| `eixos.get_area(grafico, x_range=(a, b))` | região entre a curva e o eixo x |
| `eixos.plot_derivative_graph(grafico)` | gráfico da derivada (numérica) |
| `eixos.slope_of_tangent(x, grafico)` | inclinação da tangente em `x` |

In [9]:
%%manim -qm -v WARNING AxesDetalhes
class AxesDetalhes(Scene):
    def construct(self):
        f = lambda x: x**3/3 - x
        eixos = Axes(
            x_range=[-3, 3, 1], y_range=[-2, 2, 1],               # [mín, máx, passo]: o passo define as marcas
            x_length=9, y_length=5,                               # tamanho na tela (unidades da tela, não do gráfico)
            tips=False,                                           # sem setas nas pontas
            axis_config={"include_numbers": True, "font_size": 24},   # números nos eixos, em fonte menor
            y_axis_config={"numbers_to_include": [-2, -1, 1, 2]},     # só no eixo y: omite o zero (evita número em cima da origem)
        )
        rotulos = eixos.get_axis_labels(x_label=MathTex("x"), y_label=MathTex("y"))   # nomes dos eixos
        grafico = eixos.plot(f, x_range=[-2.2, 2.2], color=YELLOW, stroke_width=5)    # stroke_width: espessura da curva
        rotulo_f = eixos.get_graph_label(grafico, label="f(x)", x_val=2, direction=UR)  # rótulo junto à curva (em x=2, acima à direita)
        self.play(Create(eixos), Write(rotulos))
        self.play(Create(grafico), Write(rotulo_f))

        ponto = eixos.i2gp(1.5, grafico)                          # i2gp: ponto DO GRÁFICO em x = 1,5 (já em coordenadas da tela)
        guia_v = eixos.get_vertical_line(ponto, color=BLUE, line_func=DashedLine)    # linha tracejada do eixo x até o ponto
        guia_h = eixos.get_horizontal_line(ponto, color=BLUE, line_func=DashedLine)  # e do eixo y até o ponto
        area = eixos.get_area(grafico, x_range=(0, 1.5), color=BLUE, opacity=0.3)    # área entre a curva e o eixo x em [0; 1,5]
        self.play(Create(guia_v), Create(guia_h), FadeIn(Dot(ponto, color=RED)))
        self.play(FadeIn(area))

        derivada = eixos.plot_derivative_graph(grafico, color=GREEN, x_range=[-1.7, 1.7])   # f' numérica (verde); x_range: f' cresce rápido e sairia dos eixos
        self.play(Create(derivada), run_time=2)
        m = eixos.slope_of_tangent(1.5, grafico)                          # inclinação da tangente em x = 1,5
        texto = Text(f"f'(1,5) ≈ {m:.2f}".replace(".", ","), font_size=30, color=GREEN).to_corner(UL)   # vírgula decimal; compare com f'(1,5) = 1,25
        self.play(Write(texto))
        self.wait(2)
        # Mais opções:
        #   eixos.get_lines_to_point(ponto)                  # as duas linhas (vertical + horizontal) de uma vez
        #   eixos.plot_antiderivative_graph(grafico)         # gráfico de uma primitiva (numérica)
        #   eixos.get_riemann_rectangles(...)                # retângulos de Riemann (Exemplo 2)
        #   eixos.get_secant_slope_group(x, grafico, dx=0.5) # "secante pronta", com Δx e Δy rotulados
        #   NumberPlane(...)                                 # eixos + malha de fundo; ThreeDAxes(...) para 3D
        #   NumberLine(x_range=[-5, 5, 1], length=10)        # só uma reta numérica (limites, intervalos)

Manim Community v0.22.0

**Detalhes do `Axes`**
* Escolha `x_length` e `y_length` proporcionais aos intervalos, para que a figura não fique "achatada"
* **Sempre** passe `x_range` no `plot`; funções como $1/x$ ou $\tan x$ explodem fora do domínio
* Guarde `eixos` e `grafico` em variáveis: quase tudo (`c2p`, `get_area`, tangente) parte deles
* Para tudo que depende de um valor que muda: `always_redraw` (Valores dinâmicos, adiante)

## <span style="color:#146dcc">■</span> Funções especiais: assíntotas e saltos

In [10]:
%%manim -qm -v WARNING FuncoesEspeciais
class FuncoesEspeciais(Scene):
    def construct(self):
        # Eixos pequenos, lado a lado (arrange organiza o grupo todo)
        e1 = Axes(x_range=[-3, 3], y_range=[-4, 4], x_length=5.5, y_length=4)
        e2 = Axes(x_range=[-3, 3], y_range=[-4, 4], x_length=5.5, y_length=4)
        VGroup(e1, e2).arrange(RIGHT, buff=0.8)

        # 1/x tem assíntota em x = 0: desenhamos DOIS pedaços, parando onde |1/x| = 4 (o limite do eixo y)
        g1 = VGroup(e1.plot(lambda x: 1/x, x_range=[-3, -0.25], color=YELLOW),
                    e1.plot(lambda x: 1/x, x_range=[0.25, 3], color=YELLOW))
        # tan x tem assíntotas em ±π/2; paramos em |x| = 1,32, onde |tan x| ≈ 4 (senão a curva sai dos eixos)
        g2 = e2.plot(np.tan, x_range=[-1.32, 1.32], color=GREEN)
        t1 = MathTex(r"y = \frac{1}{x}").next_to(e1, UP)
        t2 = MathTex(r"y = \tan x").next_to(e2, UP)
        self.play(Create(e1), Create(e2), Write(t1), Write(t2))   # eixos e títulos
        self.play(Create(g1), Create(g2), run_time=2)              # as curvas são traçadas juntas
        self.wait()
        # Opções de plot úteis:
        #   discontinuities=[a, b], dt=0.01          # onde a função é descontínua (alternativa aos pedaços, mas sem limitar a altura)
        #   use_smoothing=False                      # não suaviza (retas "retas", cantos vivos: |x|)
        #   x_range=[a, b, passo]                    # com passo = distância entre amostras (curvas "serrilhadas" ficam suaves com passo menor)
        #   stroke_width=6, stroke_opacity=0.8       # estilo da linha
        #   e1.plot_parametric_curve(lambda t: np.array([np.cos(t), np.sin(t), 0]), t_range=[0, TAU])  # curvas paramétricas

Manim Community v0.22.0

## <span style="color:#146dcc">■</span> Limites laterais

In [11]:
%%manim -qm -v WARNING LimitesLaterais
class LimitesLaterais(Scene):
    def construct(self):
        eixos = Axes(x_range=[-1, 4, 1], y_range=[-1, 4, 1], x_length=8, y_length=5,
                     axis_config={"include_numbers": True})

        def f(x):                                              # função definida POR PARTES (def, pois lambda não comporta bem o if)
            return x + 1 if x < 2 else x - 1                   # esquerda: x+1 -> 3 em x=2; direita: x-1 -> 1 em x=2

        esquerda = eixos.plot(f, x_range=[-1, 1.999], color=YELLOW)   # pedaço à esquerda de 2 (para ANTES do salto)
        direita = eixos.plot(f, x_range=[2, 4], color=YELLOW)    # pedaço à direita (a partir de 2); evite o nome dir, que já é uma função do Python
        # Bolinhas "aberta" e "fechada": convenção para extremos incluídos/excluídos
        aberta = Circle(radius=0.09, color=YELLOW, fill_color=config.background_color, fill_opacity=1).move_to(eixos.c2p(2, 3))   # "furo" com a cor do fundo: f NÃO vale 3 em x = 2
        fechada = Dot(eixos.c2p(2, 1), color=YELLOW)                                                             # f(2) = 1
        tracejada = eixos.get_vertical_line(eixos.c2p(2, 3), line_func=DashedLine, color=GREY)

        lim_esq = MathTex(r"\lim_{x \to 2^-} f(x) = 3", color=YELLOW, font_size=36)
        lim_dir = MathTex(r"\lim_{x \to 2^+} f(x) = 1", color=YELLOW, font_size=36)
        conclusao = MathTex(r"\text{os limites laterais diferem}", font_size=32)
        conclusao_2 = MathTex(r"\Rightarrow \nexists \lim_{x \to 2} f(x)", color=RED, font_size=34)
        textos = VGroup(lim_esq, lim_dir, conclusao, conclusao_2).arrange(DOWN, aligned_edge=LEFT).to_corner(UL)

        self.add(eixos)
        self.play(Create(esquerda), Create(direita), run_time=2)
        self.play(FadeIn(aberta), FadeIn(fechada), Create(tracejada))
        for t in textos:
            self.play(Write(t))
        self.wait(2)

Manim Community v0.22.0

## <span style="color:#146dcc">■</span> Funções anônimas

* Para funções curtas, usadas uma única vez, o Python oferece a forma `lambda`
* As duas definições abaixo são equivalentes:

In [12]:
def g(x):
    return 2*x + 1

h = lambda x: 2*x + 1

g(3), h(3)

(7, 7)

* Assim, `eixos.plot(lambda x: 2*x + 1)` desenha a reta $y = 2x + 1$ sem precisarmos dar um nome à função
* Também em `sorted`, `max`, ... é comum passarmos uma `lambda`

## <span style="color:#146dcc">■</span> Definindo funções para os gráficos

O `plot` aceita **qualquer função Python** que receba um número e devolva um número:

| jeito | exemplo | quando usar |
|---|---|---|
| `def` | `def f(x): return x**3/3 - x` | funções longas, por partes, reaproveitadas |
| `lambda` | `lambda x: x**2` | funções curtas, usadas uma vez |
| função do NumPy | `np.sin`, `np.exp`, `np.log`, `np.sqrt`, `np.tan` | funções elementares prontas |
| composição | `lambda x: np.exp(-x**2)*np.cos(3*x)` | combinando as anteriores |
| por partes | `def f(x): return x**2 if x < 1 else 2 - x` | funções definidas por sentenças |

**Cuidados:** use `**` para potência (e não `^`), `np.sqrt`/`np.log` (e não `math.`, para aceitar arrays) e `x_range` para evitar domínios fora de definição.

## <span style="color:#146dcc">■</span> Valores dinâmicos

* Como animar o gráfico de $y = ax^2$ enquanto $a$ varia?
* `ValueTracker(v)`: guarda um número que pode ser **animado**
  - leitura: `a.get_value()`
  - animação: `self.play(a.animate.set_value(3))`
* `always_redraw(func)`: refaz o objeto, chamando `func`, **a cada quadro** (assim ele "acompanha" o valor)

In [13]:
%%manim -qm -v WARNING ParametroA
class ParametroA(Scene):
    def construct(self):
        a = ValueTracker(1)                            # o "controle deslizante": começa valendo 1
        eixos = Axes(x_range=[-3, 3], y_range=[-4, 4], x_length=6, y_length=6)

        def parabola():                                # esta função é chamada A CADA QUADRO por always_redraw
            return eixos.plot(lambda x: a.get_value()*x**2,   # lê o valor atual de a (muda durante a animação)
                              x_range=[-1.4, 1.4], color=YELLOW)

        grafico = always_redraw(parabola)              # sempre refeito conforme a muda; passe a função SEM parênteses

        rotulo = MathTex("a =").to_corner(UL)
        valor = DecimalNumber(1, num_decimal_places=2).next_to(rotulo, RIGHT)   # número exibido com 2 casas decimais
        # add_updater: a cada quadro, atualiza o número mostrado para o valor atual de a (m é o próprio objeto)
        valor.add_updater(lambda m: m.set_value(a.get_value()))
        legenda = MathTex("y = ax^2").next_to(rotulo, DOWN, aligned_edge=LEFT)

        self.add(eixos, grafico, rotulo, valor, legenda)    # add: tudo já começa na tela
        self.play(a.animate.set_value(2), run_time=2)       # a: 1 → 2 (parábola mais fechada)
        self.play(a.animate.set_value(-2), run_time=3)      # a: 2 → -2 (vira para baixo ao passar por 0)
        self.play(a.animate.set_value(0.5), run_time=2)
        self.wait()

Manim Community v0.22.0

* `add_updater(func)` é a alternativa a `always_redraw` quando o objeto apenas **muda** (aqui, o valor exibido), em vez de ser refeito
* Regra prática: o objeto muda de **forma** (gráfico, reta) → `always_redraw`; muda só de **valor/posição** → `add_updater`

## <span style="color:#146dcc">■</span> Valores dinâmicos: panorama

| ferramenta | uso | exemplo |
|---|---|---|
| `ValueTracker(v)` | guarda um número **animável** | `t = ValueTracker(0)`; `t.get_value()`; `t.animate.set_value(3)`; `t.increment_value(1)` |
| `always_redraw(func)` | **refaz** o objeto a cada quadro | gráfico, reta tangente, retângulos de Riemann |
| `obj.add_updater(func)` | **altera** o objeto a cada quadro | número mostrado, posição; `func(m)` ou `func(m, dt)` |
| `UpdateFromFunc(obj, func)` | animação que chama `func(obj)` a cada quadro | `self.play(UpdateFromFunc(...))` |
| `UpdateFromAlphaFunc(obj, func)` | idem, recebendo o progresso `alpha` (0 a 1) | interpolações personalizadas |
| `Variable(valor, rótulo)` | rótulo + número + `ValueTracker` já ligados | `var.tracker.animate.set_value(2)` |
| `obj.clear_updaters()` | **remove** os updaters | antes de mover o objeto "na mão" |
| `obj.suspend_updating()` / `resume_updating()` | pausa / retoma o updater | congelar um número por instantes |

In [14]:
%%manim -qm -v WARNING VariavelEUpdaters
class VariavelEUpdaters(Scene):
    def construct(self):
        # Variable: já traz o rótulo, o número e o ValueTracker (var.tracker) ligados entre si
        var = Variable(0.5, MathTex("a"), num_decimal_places=2).to_corner(UL)
        eixos = Axes(x_range=[-3, 3], y_range=[-1, 5], x_length=6, y_length=5).shift(RIGHT)
        # always_redraw: a parábola é REFEITA a cada quadro, usando o valor atual do tracker
        parabola = always_redraw(lambda: eixos.plot(lambda x: var.tracker.get_value()*x**2,
                                                    x_range=[-1.5, 1.5], color=YELLOW))   # com a = 2, y chega a 4,5 (o eixo vai até 5)
        self.add(var, eixos, parabola)
        self.play(var.tracker.animate.set_value(2), run_time=2)         # anima o tracker: tudo que depende dele acompanha

        # updater com dt (tempo desde o quadro anterior, em segundos): movimento com velocidade CONSTANTE
        ponto = Dot(eixos.c2p(-2, 0), color=RED)
        ponto.add_updater(lambda m, dt: m.shift(dt*RIGHT))              # 1 unidade da tela por segundo, para a direita
        self.add(ponto)
        self.wait(2)                                                    # durante este wait, o updater faz o ponto andar
        ponto.clear_updaters()                                          # para o movimento (senão ele continuaria andando)
        self.wait()
        # Outras ferramentas:
        #   self.play(UpdateFromAlphaFunc(obj, lambda m, alpha: m.set_opacity(alpha)))  # interpolação personalizada
        #   var.tracker.increment_value(1)           # soma 1 ao valor (sem animar)
        #   n = DecimalNumber(0); n.add_updater(lambda m: m.set_value(t.get_value()), call_updater=True)  # call_updater: atualiza já na criação

Manim Community v0.22.0

## <span style="color:#146dcc">■</span> Exemplo 1: da secante à tangente

A derivada é o **limite** das inclinações das secantes:
$$
f'(x_0) = \lim_{h \to 0} \frac{f(x_0 + h) - f(x_0)}{h}.
$$
Fixamos $x_0 = 1{,}5$ e fazemos $h$ diminuir de $1$ até quase $0$: a inclinação da secante converge para $f'(1{,}5) = 1{,}5^2 - 1 = 1{,}25$.

In [15]:
%%manim -qm -v WARNING SecanteTangente
class SecanteTangente(Scene):
    def construct(self):
        f = lambda x: x**3/3 - x                       # a função (lambda, pois é curta)
        eixos = Axes(x_range=[-3, 3, 1], y_range=[-3, 3, 1], x_length=8, y_length=5,
                     axis_config={"include_numbers": True}).to_edge(RIGHT)   # eixos na metade direita da tela
        grafico = eixos.plot(f, x_range=[-2.5, 2.5], color=YELLOW)

        x0 = 1.5                                       # ponto fixo (P)
        h = ValueTracker(1)                            # passo h: o valor que vai ficar cada vez menor

        def inclinacao():                              # inclinação da secante para o h atual (razão incremental)
            return (f(x0 + h.get_value()) - f(x0)) / h.get_value()

        def secante():                                 # reta por P e Q, refeita a cada quadro
            m = inclinacao()
            return eixos.plot(lambda t: f(x0) + m*(t - x0),       # reta: y = f(x0) + m(t - x0)
                              x_range=[x0 - 1.2, x0 + 1.2], color=RED)

        def ponto_q():                                 # ponto Q = (x0 + h, f(x0 + h)), que "desliza" até P
            return Dot(eixos.c2p(x0 + h.get_value(), f(x0 + h.get_value())), color=RED)

        reta = always_redraw(secante)                  # always_redraw: formas que dependem de h
        Q = always_redraw(ponto_q)
        P = Dot(eixos.c2p(x0, f(x0)), color=WHITE)     # P não muda: não precisa de always_redraw

        # painel com os valores de h e da inclinação, atualizados por updaters (só mudam de valor)
        txt_h = MathTex("h =")
        num_h = DecimalNumber(1, num_decimal_places=3)
        num_h.add_updater(lambda m: m.set_value(h.get_value()))
        txt_m = MathTex(r"m_{sec} =")
        num_m = DecimalNumber(0, num_decimal_places=3)
        num_m.add_updater(lambda m: m.set_value(inclinacao()))
        painel = VGroup(VGroup(txt_h, num_h).arrange(RIGHT),
                        VGroup(txt_m, num_m).arrange(RIGHT)).arrange(DOWN, aligned_edge=LEFT).to_corner(UL)

        self.add(eixos, grafico, P)
        self.play(Create(reta), FadeIn(Q), FadeIn(painel))
        self.play(h.animate.set_value(0.01), run_time=6)       # h → 0: a secante gira e vira a tangente
        self.wait()
        limite = MathTex(r"f'(1{,}5) = 1{,}25").next_to(painel, DOWN, buff=0.8, aligned_edge=LEFT)
        self.play(Write(limite), Circumscribe(num_m))          # compara com o valor exato
        self.wait(2)

Manim Community v0.22.0

* O mesmo código, trocando só `f` e `x0`, anima a derivada de **qualquer** função
* Cuidado: `h` **não** pode chegar a $0$ (divisão por zero): por isso paramos em $0{,}01$

## <span style="color:#146dcc">■</span> Exemplo 2: soma de Riemann

Para $f(x) = x^2$ em $[0, 2]$, a integral vale $\displaystyle\int_0^2 x^2\,dx = \frac{8}{3} \approx 2{,}667$.

* `eixos.get_riemann_rectangles(grafico, x_range, dx)` cria os retângulos de largura `dx`
* Com um `ValueTracker` para `dx` e `always_redraw`, vemos a soma convergir

In [16]:
%%manim -qm -v WARNING SomaDeRiemann
class SomaDeRiemann(Scene):
    def construct(self):
        f = lambda x: x**2
        eixos = Axes(x_range=[0, 2.5, 0.5], y_range=[0, 4.5, 1], x_length=6, y_length=5,
                     axis_config={"include_numbers": True}).to_edge(LEFT)
        grafico = eixos.plot(f, x_range=[0, 2.1], color=YELLOW)

        dx = ValueTracker(0.5)                         # largura dos retângulos: vai diminuir

        def retangulos():                              # refeito a cada quadro com o dx atual
            return eixos.get_riemann_rectangles(
                grafico, x_range=[0, 2], dx=dx.get_value(),     # x_range: intervalo de integração [0, 2]
                stroke_width=0.5, fill_opacity=0.7)             # contorno fino e preenchimento semitransparente

        def soma():                                    # a mesma soma, calculada em Python (extremo esquerdo)
            d = dx.get_value()
            xs = np.arange(0, 2, d)                    # pontos x_i: 0, d, 2d, ... (menores que 2)
            return float(np.sum(xs**2 * d))            # Σ f(x_i)·Δx

        barras = always_redraw(retangulos)

        texto_dx = MathTex(r"\Delta x =")
        num_dx = DecimalNumber(0.5, num_decimal_places=3)
        num_dx.add_updater(lambda m: m.set_value(dx.get_value()))
        texto_s = MathTex(r"\sum f(x_i)\,\Delta x =")
        num_s = DecimalNumber(0, num_decimal_places=3)
        num_s.add_updater(lambda m: m.set_value(soma()))
        painel = VGroup(VGroup(texto_dx, num_dx).arrange(RIGHT),
                        VGroup(texto_s, num_s).arrange(RIGHT)).arrange(DOWN, aligned_edge=LEFT)
        painel.to_corner(UR)

        alvo = MathTex(r"\int_0^2 x^2\,dx = \frac{8}{3} \approx 2{,}667", color=YELLOW).next_to(painel, DOWN, buff=0.8, aligned_edge=LEFT)

        self.add(eixos, grafico, barras, painel)
        self.wait()
        self.play(dx.animate.set_value(0.05), run_time=8)   # dx: 0,5 → 0,05 (de 4 para 40 retângulos)
        self.play(Write(alvo))
        self.wait(2)

Manim Community v0.22.0

* As somas de extremo esquerdo de uma função crescente ficam **abaixo** da integral e se aproximam dela por baixo
* Mude `x_range`, `f` e o intervalo para animar outra integral

## <span style="color:#146dcc">■</span> Exemplo 3: o gráfico da derivada

Enquanto $x_0$ percorre o gráfico de $f$, a **inclinação** da tangente vira a **altura** do ponto $(x_0, f'(x_0))$, que desenha o gráfico de $f'$ ao lado. É a ideia central da derivada como **função**.

In [17]:
%%manim -qm -v WARNING GraficoDaDerivada
class GraficoDaDerivada(Scene):
    def construct(self):
        f = lambda x: x**3/3 - x                                   # função
        df = lambda x: x**2 - 1                                    # derivada (exata)
        e1 = Axes(x_range=[-3, 3], y_range=[-2, 3], x_length=6, y_length=4)    # y até 3: f'(±2) = 3 precisa caber
        e2 = Axes(x_range=[-3, 3], y_range=[-2, 3], x_length=6, y_length=4)
        VGroup(e1, e2).arrange(RIGHT, buff=0.7)                    # lado a lado (muda a posição dos DOIS eixos)
        g1 = e1.plot(f, x_range=[-2.2, 2.2], color=YELLOW)
        t1 = MathTex("f(x)").next_to(e1, UP)
        t2 = MathTex("f'(x)", color=GREEN).next_to(e2, UP)

        x0 = ValueTracker(-2)                                    # abscissa que vai percorrer a curva

        def tangente():                                            # reta tangente (refeita a cada quadro)
            x = x0.get_value()
            return e1.plot(lambda t: f(x) + df(x)*(t - x), x_range=[x - 0.5, x + 0.5], color=RED)   # segmento curto: a inclinação chega a 3

        ponto_f = always_redraw(lambda: Dot(e1.c2p(x0.get_value(), f(x0.get_value())), color=RED))      # ponto na curva de f
        ponto_df = always_redraw(lambda: Dot(e2.c2p(x0.get_value(), df(x0.get_value())), color=GREEN))  # ponto (x0, f'(x0))
        # TracedPath: deixa o "rastro" do ponto: o gráfico de f' aparece desenhado pelo movimento
        rastro = TracedPath(ponto_df.get_center, stroke_color=GREEN, stroke_width=4)

        self.add(e1, e2, g1, t1, t2, rastro)
        self.add(always_redraw(tangente), ponto_f, ponto_df)
        self.play(x0.animate.set_value(2), run_time=8, rate_func=linear)   # linear: o rastro sai com velocidade uniforme
        self.wait(2)

Manim Community v0.22.0

* Observe: onde a tangente é **horizontal**, o ponto de $f'$ cruza o eixo $x$ ($x = \pm 1$)
* Onde $f$ **cresce**, $f' > 0$ (ponto acima do eixo); onde **decresce**, $f' < 0$

## <span style="color:#146dcc">■</span> Exercícios

1. Anime o gráfico de $y = \sin(ax)$, com $a$ variando de $1$ a $3$, exibindo o valor de $a$ (use `np.sin`).
2. Anime $y = (x - m)^2 + k$ movendo o vértice $(m, k)$ ao longo da reta $y = x$ (dois `ValueTracker`s, ou um só com $m = k$).


# Parte 5: estudo de caso e oficina

<div style="border-left:10px solid #f06418; background:#fff0e7; color:#082956; border-radius:12px; padding:14px 20px; font-family:Sora, Manrope, system-ui, -apple-system, 'Segoe UI', sans-serif; font-size:17px">Estudo de caso, boas práticas e oficina: animando questões de Cálculo.</div>

## <span style="color:#f06418">■</span> Uma questão de Cálculo

> Seja $f(x) = \dfrac{x^3}{3} - x$. Determine os **pontos críticos** de $f$ e classifique-os (máximo local, mínimo local ou nenhum dos dois).

* Tema: aplicações da derivada: $f'(x) = 0$ e variação do sinal de $f'$
* Resposta esperada: máximo local em $x = -1$, com $f(-1) = \tfrac{2}{3}$; mínimo local em $x = 1$, com $f(1) = -\tfrac{2}{3}$

## <span style="color:#f06418">■</span> Roteiro da animação

Antes do código, o **roteiro** (*storyboard*):
1. Título e enunciado
2. Derivada e equação $f'(x) = 0$ (passagens algébricas com `TransformMatchingTex`)
3. Gráfico de $f$ e **tangente que percorre a curva**, com o valor de $f'(x_0)$
4. Conclusão: sinal de $f'$ e classificação dos pontos críticos

* Cada etapa vira um **método** da cena
* O `construct` apenas chama as etapas, na ordem
* **Por quê?** Código curto e organizado: se a etapa 3 falhar, você sabe onde procurar

## <span style="color:#f06418">■</span> Organizando o código

In [18]:
# Funções matemáticas FORA da classe: assim todos os métodos (e outras cenas) podem usá-las
def f(x):
    return x**3/3 - x

def df(x):
    return x**2 - 1                                  # derivada exata de f

class PontosCriticos(Scene):
    def construct(self):                             # o roteiro: só chama as etapas, na ordem
        self.enunciado()
        self.derivar()
        self.grafico()
        self.conclusao()

* `self.enunciado()`: chama um método **nosso**, definido na própria classe (como `self.play`, mas escrito por nós)
* Os métodos compartilham objetos pela própria cena: guardamos em `self.algo` o que outra etapa vai reutilizar

## <span style="color:#f06418">■</span> Etapa 1: enunciado

In [19]:
def enunciado(self):
    titulo = Text("Pontos críticos", font_size=40).to_edge(UP, buff=0.3)       # título na borda superior
    # Tex: texto LaTeX com fórmulas entre $ $; font_size menor, pois o texto é longo
    self.questao = VGroup(
        MathTex(r"f(x) = \frac{x^3}{3} - x"),
        Tex(r"Determine os pontos críticos de $f$", font_size=30),
        Tex(r"e classifique-os.", font_size=30),
    ).arrange(DOWN, aligned_edge=LEFT)                                # empilha, alinhando à esquerda
    self.questao.next_to(titulo, DOWN, buff=0.3).to_edge(LEFT)        # self.questao: guardado para as outras etapas posicionarem relativamente a ele
    self.play(Write(titulo))
    self.play(Write(self.questao), run_time=2)

PontosCriticos.enunciado = enunciado        # acrescenta o método à classe (só necessário porque estamos dividindo a classe em várias células)

* Em um arquivo `.py`, os métodos ficam **dentro** da classe; aqui, a linha `PontosCriticos.enunciado = enunciado` os acrescenta célula a célula, para caber nos slides

## <span style="color:#f06418">■</span> Etapa 2: derivada

In [20]:
def derivar(self):
    # cada passo tem as mesmas partes-chave ("=") e muda o resto: o TransformMatchingTex move o que é igual
    passos = [
        MathTex(r"f'(x)", "=", r"x^2 - 1"),                           # derivada de x³/3 - x
        MathTex(r"x^2 - 1", "=", "0"),                                # ponto crítico: f'(x) = 0
        MathTex(r"x^2", "=", "1"),                                    # isola x²
        MathTex(r"x", "=", r"\pm 1"),                                 # raiz quadrada: dois pontos críticos
    ]
    for p in passos:
        p.next_to(self.questao, DOWN, buff=0.4, aligned_edge=LEFT)   # todos no mesmo lugar, abaixo do enunciado
    atual = passos[0]
    self.play(Write(atual))
    for proximo in passos[1:]:
        self.wait(0.8)
        self.play(TransformMatchingTex(atual, proximo))
        atual = proximo
    self.derivacao = atual                                           # guarda o último para posicionar a conclusão
    self.play(Circumscribe(atual, color=YELLOW))

PontosCriticos.derivar = derivar

## <span style="color:#f06418">■</span> Etapa 3: gráfico e tangente

In [21]:
def grafico(self):
    self.eixos = Axes(x_range=[-3, 3, 1], y_range=[-2, 2, 1], x_length=6.5, y_length=4.5,
                      axis_config={"include_numbers": True}).to_edge(RIGHT).shift(0.3*DOWN)
    curva = self.eixos.plot(f, x_range=[-2.2, 2.2], color=YELLOW)    # usa a f definida fora da classe
    self.play(Create(self.eixos), Create(curva))

    x0 = ValueTracker(-2)                                            # abscissa do ponto de tangência
    def tangente():                                                  # reta tangente: y = f(x0) + f'(x0)(x - x0)
        x = x0.get_value()
        return self.eixos.plot(lambda t: f(x) + df(x)*(t - x),
                               x_range=[x - 0.7, x + 0.7], color=RED)
    def ponto():                                                     # ponto da curva
        x = x0.get_value()
        return Dot(self.eixos.c2p(x, f(x)), color=RED)
    reta, p = always_redraw(tangente), always_redraw(ponto)          # refeitos a cada quadro

    valor = DecimalNumber(df(-2), num_decimal_places=2)              # inclinação mostrada
    valor.add_updater(lambda m: m.set_value(df(x0.get_value())))     # só muda de valor: updater
    painel = VGroup(MathTex(r"f'(x_0) ="), valor).arrange(RIGHT).next_to(self.eixos, UP)

    self.play(Create(reta), FadeIn(p), FadeIn(painel))
    self.play(x0.animate.set_value(2), run_time=7, rate_func=linear)  # linear: velocidade constante, para o aluno acompanhar
    self.wait()
    self.play(FadeOut(reta, p, painel))                              # tira a tangente, o ponto e o painel; ficam só eixos e curva
    self.remove(reta, p)                                             # remove os objetos "vivos" (que têm updaters) da cena de vez

PontosCriticos.grafico = grafico

## <span style="color:#f06418">■</span> Etapa 4: conclusão

In [22]:
def conclusao(self):
    # marcas nos pontos críticos, dentro do gráfico (c2p converte coordenadas dos eixos em pontos da tela)
    maximo = Dot(self.eixos.c2p(-1, f(-1)), color=GREEN)
    minimo = Dot(self.eixos.c2p(1, f(1)), color=RED)
    r_max = MathTex(r"\text{máx.}", font_size=30, color=GREEN).next_to(maximo, UP, buff=0.15)
    r_min = MathTex(r"\text{mín.}", font_size=30, color=RED).next_to(minimo, DOWN, buff=0.15)

    # sinal de f' (resumo): três linhas
    linhas = VGroup(
        MathTex(r"x < -1:\ f' > 0 \Rightarrow f \text{ cresce}", font_size=28),
        MathTex(r"-1 < x < 1:\ f' < 0 \Rightarrow f \text{ decresce}", font_size=28),
        MathTex(r"x > 1:\ f' > 0 \Rightarrow f \text{ cresce}", font_size=28),
    ).arrange(DOWN, aligned_edge=LEFT).next_to(self.derivacao, DOWN, buff=0.3, aligned_edge=LEFT)
    for linha in linhas:
        self.play(Write(linha))                                      # uma linha por vez, no ritmo da explicação
    self.play(FadeIn(maximo, r_max), FadeIn(minimo, r_min))

    resposta = VGroup(
        MathTex(r"x = -1:\ \text{máximo local},\ f(-1) = \tfrac{2}{3}", font_size=28, color=GREEN),
        MathTex(r"x = 1:\ \text{mínimo local},\ f(1) = -\tfrac{2}{3}", font_size=28, color=RED),
    ).arrange(DOWN, aligned_edge=LEFT).next_to(linhas, DOWN, buff=0.3, aligned_edge=LEFT)
    self.play(Write(resposta))
    self.wait(2)

PontosCriticos.conclusao = conclusao

## <span style="color:#f06418">■</span> Resultado

Com a classe completa, renderizamos a cena:

In [23]:
%%manim -qm -v WARNING PontosCriticos
# Nesta célula não há código novo: o %%manim renderiza a classe PontosCriticos, já definida nas células anteriores.

Manim Community v0.22.0

* Em um arquivo `.py`, os métodos ficariam dentro do próprio `class PontosCriticos(Scene):`
* Para ajustar uma etapa, **reexecute** só a célula dela e depois a de renderização (use `-ql` enquanto testa)
* **Atenção:** se reexecutar a célula `class PontosCriticos`, os métodos anexados se perdem; reexecute também as células das quatro etapas


## <span style="color:#f06418">■</span> Boas práticas

* Escreva o **roteiro** antes do código
* Divida a cena em métodos, um por etapa
* Teste com `-ql`; renderize a versão final com `-qh`
* Use `self.wait()` para dar tempo de leitura
* Prefira posicionamento **relativo** (`next_to`, `align_to`) a coordenadas fixas
* Funções do problema (`f`, `df`) ficam **fora** da cena: trocar a função troca a animação inteira
* Comente o código: ele também é material didático

## <span style="color:#f06418">■</span> Oficina

Em duplas, escolha **uma questão de Cálculo** (ou proponha outra) e anime a resolução:

| tema | ideia de animação |
|---|---|
| **Limite** | $\lim_{x\to 0}\frac{\sin x}{x} = 1$: valores de $x$ tendendo a $0$ e o quociente em um `DecimalNumber` |
| **Derivada** | tangente a $f(x) = x^2$ em $x_0 = 1$; regra do produto com `TransformMatchingTex` |
| **Taxa relacionada** | escada que escorrega, ou círculo cujo raio cresce: $A'(t)$ |
| **Integral** | área sob $y = \sin x$ em $[0, \pi]$ por Riemann (`get_riemann_rectangles`) e valor exato $2$ |
| **Teorema Fundamental** | área acumulada $A(x) = \int_0^x f(t)\,dt$ crescendo conforme $x$ varia |
| **Série** | polinômios de Taylor de $e^x$ ou $\cos x$ |

1. Escolha o tema e **escreva o roteiro** em 3 a 5 etapas
2. Implemente a cena, **um método por etapa** (use o modelo da próxima célula)
3. Teste com `-ql`; renderize em qualidade `-qh`
4. Apresente à turma e explique **uma** escolha de código (por que `always_redraw`? por que `next_to`?)

## <span style="color:#f06418">■</span> Modelo para a oficina

In [24]:
%%manim -ql -v WARNING MinhaQuestao
# Copie esta célula, mude o nome da classe (aqui e na 1ª linha) e preencha os "TODO".

def f(x):                                    # TODO: a função do seu problema
    return x**2

class MinhaQuestao(Scene):
    def construct(self):                     # o roteiro: uma chamada por etapa
        self.enunciado()
        self.resolucao()
        self.conclusao()

    def enunciado(self):
        # TODO: título e enunciado (Text / MathTex / Tex), posicionados com to_edge / next_to
        titulo = Text("Meu título", font_size=40).to_edge(UP)
        self.play(Write(titulo))

    def resolucao(self):
        # TODO: gráfico (Axes + plot) ou passagens algébricas (TransformMatchingTex);
        #       use ValueTracker + always_redraw se algo precisa variar
        eixos = Axes(x_range=[-3, 3], y_range=[0, 9], x_length=6, y_length=4)
        self.play(Create(eixos), Create(eixos.plot(f, color=YELLOW)))

    def conclusao(self):
        # TODO: resultado final em destaque (Circumscribe, Indicate, mudança de cor)
        resposta = MathTex(r"\text{resposta}").to_edge(DOWN)
        self.play(Write(resposta))
        self.wait(2)

Manim Community v0.22.0

## <span style="color:#f06418">■</span> Modelos prontos

Cenas completas para **adaptar** na oficina. Em aula, mostramos só o vídeo de cada uma; o código fica de referência.

## <span style="color:#f06418">■</span> Modelo pronto 1: polinômios de Taylor de $\sin x$

Cada polinômio de Taylor "abraça" o seno em um intervalo maior:
$$
\sin x \approx x,\quad x - \frac{x^3}{3!},\quad x - \frac{x^3}{3!} + \frac{x^5}{5!},\quad \dots
$$

In [25]:
%%manim -qm -v WARNING TaylorSeno
from math import factorial                          # factorial(n) = n!; importamos para escrever o termo geral

class TaylorSeno(Scene):
    def construct(self):
        eixos = Axes(x_range=[-4, 4, 1], y_range=[-2.5, 2.5, 1], x_length=10, y_length=5,   # até 2,5: P3 chega a ±2,3 nas pontas

                     axis_config={"include_numbers": True})
        seno = eixos.plot(np.sin, x_range=[-4, 4], color=YELLOW)    # np.sin: função pronta do NumPy (dispensa def/lambda)
        self.play(Create(eixos), Create(seno))

        def taylor(n):                               # polinômio de grau n: soma dos termos ímpares (-1)^k x^(2k+1)/(2k+1)!
            return lambda x: sum((-1)**k * x**(2*k + 1) / factorial(2*k + 1) for k in range(n))

        curva = eixos.plot(taylor(1), x_range=[-3.2, 3.2], color=RED)   # começa com P1(x) = x
        rotulo = MathTex(r"P_1(x)", color=RED).to_corner(UL)
        self.play(Create(curva), Write(rotulo))
        for n in [2, 3, 4]:                          # n termos: graus 3, 5 e 7
            nova = eixos.plot(taylor(n), x_range=[-3.2, 3.2], color=RED)
            grau = 2*n - 1
            novo_rotulo = MathTex(rf"P_{{{grau}}}(x)", color=RED).to_corner(UL)   # rf"": raw + f-string; {{ }} viram { } no LaTeX
            self.play(Transform(curva, nova), Transform(rotulo, novo_rotulo), run_time=2)
            self.wait(0.5)
        self.wait(2)

Manim Community v0.22.0

* `taylor(n)` **devolve uma função** (uma `lambda`): é o que o `eixos.plot` espera
* `Transform(curva, nova)` deforma a curva atual até a nova: dá para **ver** a aproximação melhorar

## <span style="color:#f06418">■</span> Modelo pronto 2: limite fundamental

Mostra
$$\lim_{x \to 0} \frac{\sin x}{x} = 1$$
aproximando $x$ de $0$ pelos **dois lados** e lendo o valor da função.

In [26]:
%%manim -qm -v WARNING LimiteSeno
class LimiteSeno(Scene):
    def construct(self):
        def f(x):                                                   # sen(x)/x, definida por partes: em x = 0 a expressão é 0/0
            return np.sin(x)/x if x != 0 else 1
        eixos = Axes(x_range=[-6, 6, 2], y_range=[-0.5, 1.5, 0.5], x_length=9, y_length=4,
                     axis_config={"include_numbers": True},
                     y_axis_config={"numbers_to_include": [-0.5, 0.5]}).to_edge(DOWN)   # sem o "1.0", que ficaria em cima do furo
        grafico = eixos.plot(f, x_range=[-6, 6], color=YELLOW)
        # o ponto (0, 1) é "buraco" da função original: bolinha aberta
        buraco = Circle(radius=0.08, color=YELLOW, fill_color=config.background_color, fill_opacity=1).move_to(eixos.c2p(0, 1))   # "furo" com a cor do fundo
        self.play(Create(eixos), Create(grafico))
        self.play(FadeIn(buraco))

        x = ValueTracker(3)                                         # distância de x a 0: vai diminuir
        pto_d = always_redraw(lambda: Dot(eixos.c2p(x.get_value(), f(x.get_value())), color=RED))      # lado direito (x > 0)
        pto_e = always_redraw(lambda: Dot(eixos.c2p(-x.get_value(), f(-x.get_value())), color=BLUE))   # lado esquerdo (x < 0)

        # tabelinha de valores, atualizada por updaters (só mudam de valor)
        tx = MathTex("x =").set_color(RED); nx = DecimalNumber(3, num_decimal_places=3, color=RED)
        tf = MathTex(r"\frac{\sin x}{x} =").set_color(RED); nf = DecimalNumber(f(3), num_decimal_places=4, color=RED)
        nx.add_updater(lambda m: m.set_value(x.get_value()))
        nf.add_updater(lambda m: m.set_value(f(x.get_value())))
        painel = VGroup(VGroup(tx, nx).arrange(RIGHT), VGroup(tf, nf).arrange(RIGHT)).arrange(DOWN, aligned_edge=LEFT).to_corner(UL)

        self.add(pto_d, pto_e, painel)
        self.play(x.animate.set_value(0.01), run_time=6)            # x → 0 pelos dois lados: f(x) → 1
        self.play(Write(MathTex(r"\lim_{x \to 0} \frac{\sin x}{x} = 1").next_to(painel, RIGHT, buff=1.5)))
        self.wait(2)

Manim Community v0.22.0

## <span style="color:#f06418">■</span> Modelo pronto 3: área acumulada

O **Teorema Fundamental**: se $A(x) = \int_0^x f(t)\,dt$, então $A'(x) = f(x)$. Com $f = \cos$, a área sob a curva (à esquerda) vai desenhando $A$ (à direita), que é $\sin x$.

In [ ]:
%%manim -qm -v WARNING AreaAcumulada
from scipy.integrate import quad                                 # quad: integração numérica (já vem instalada com o Manim)

class AreaAcumulada(Scene):
    def construct(self):
        f = np.cos                                                # integrando: f(t) = cos t
        A = lambda x: quad(f, 0, x)[0]                            # A(x) = ∫_0^x f: quad devolve (valor, erro); [0] fica com o valor
        e1 = Axes(x_range=[0, 3.2, 1], y_range=[-1.2, 1.2, 1], x_length=5.5, y_length=3.5)
        e2 = Axes(x_range=[0, 3.2, 1], y_range=[-1.2, 1.2, 1], x_length=5.5, y_length=3.5)
        VGroup(e1, e2).arrange(RIGHT, buff=1)
        g1 = e1.plot(f, x_range=[0, 3.1], color=YELLOW)
        t1 = MathTex(r"f(t) = \cos t").next_to(e1, UP)
        t2 = MathTex(r"A(x) = \int_0^x f(t)\,dt", color=GREEN).next_to(e2, UP)

        x = ValueTracker(0.01)                                    # limite superior da integral
        # área: refeita a cada quadro com o x atual
        area = always_redraw(lambda: e1.get_area(g1, x_range=(0, x.get_value()), color=BLUE, opacity=0.5))
        ponto = always_redraw(lambda: Dot(e2.c2p(x.get_value(), A(x.get_value())), color=GREEN))
        rastro = TracedPath(ponto.get_center, stroke_color=GREEN, stroke_width=4)   # o gráfico de A nasce do movimento do ponto

        self.add(e1, e2, g1, t1, t2, area, ponto, rastro)
        self.play(x.animate.set_value(3.1), run_time=7, rate_func=linear)
        self.play(Write(MathTex(r"A'(x) = f(x) \Rightarrow A(x) = \sin x").to_edge(DOWN)))
        self.wait(2)

## <span style="color:#f06418">■</span> Baixando o vídeo no Colab

* O `%%manim` salva os vídeos na pasta `media/` da máquina do Colab (que **some** ao fim da sessão)
* Para guardar a sua animação, baixe o arquivo `.mp4` antes de sair:

In [ ]:
import glob                                           # glob lista arquivos que casam com um padrão
from google.colab import files                         # módulo do Colab para baixar/enviar arquivos

arquivos = sorted(glob.glob("media/jupyter/*.mp4"))    # vídeos gerados pelo %%manim (uma cópia por renderização)
print(arquivos)                                        # confira os nomes; se a lista vier vazia, use: !find media -name "*.mp4"
files.download(arquivos[-1])                           # baixa o último (o mais recente) para o seu computador

## <span style="color:#f06418">■</span> Para ir além

* Narração: o pacote [`manim-voiceover`](https://voiceover.manim.community/) sincroniza falas com as animações
* Outras cenas: `MovingCameraScene` (câmera móvel), `ThreeDScene` (3D: superfícies, integrais duplas), ...
* Cálculo em várias variáveis: `ThreeDAxes`, `Surface`
* Documentação: [docs.manim.community](https://docs.manim.community/)
* Galeria de exemplos: [docs.manim.community/en/stable/examples.html](https://docs.manim.community/en/stable/examples.html)
* Canal [3Blue1Brown](https://www.youtube.com/c/3blue1brown): a série *Essence of Calculus* é uma ótima fonte de ideias

<div style="background:#082956; color:#ffffff; border-radius:18px; padding:28px 32px; font-family:Sora, Manrope, system-ui, -apple-system, 'Segoe UI', sans-serif">
<div style="font-family:'Space Mono', 'IBM Plex Mono', ui-monospace, monospace; font-size:13px; letter-spacing:.08em; color:#f4b718">III SEPEC · FIM DO MINICURSO</div>
<div style="font-size:34px; font-weight:800; line-height:1.2; margin:10px 0 8px">Obrigado!</div>
<div style="font-size:18px; opacity:.9">Animando o Cálculo com o Manim · 26 e 27/11 · docs.manim.community</div>
<div style="margin-top:20px; font-size:15px; opacity:.85">Wilia Alberto e Vicente Sobrinho · UFCA, Campus Juazeiro do Norte</div><div style="display:flex; height:6px; border-radius:3px; overflow:hidden; margin-top:18px"><div style="flex:1; background:#57a52d"></div><div style="flex:1; background:#146dcc"></div><div style="flex:1; background:#f06418"></div><div style="flex:1; background:#f4b718"></div></div>
</div>